# Training run explorer

Visualise any run under `runs/`. Every chart adapts to whichever game produced
the log — the per-game columns (`kills`, `max_x`, `score`, …) come from the
adapter's `log_fields`, so nothing here is Zelda-specific.

Run all cells, then set `RUN` in the "Pick a run" cell to whichever you want.

```bash
uv run jupyter lab runs/explore.ipynb
```

## Setup

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.ticker import MaxNLocator

RUNS = Path.cwd() if Path.cwd().name == "runs" else Path.cwd() / "runs"

# Validated categorical palette, assigned in fixed order and never cycled.
# (dataviz check: CVD separation worst adjacent dE 9.1 protan, normal-vision 22.9)
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
          "#e87ba4", "#008300", "#4a3aa7", "#8a5a2b"]
SURFACE, INK, INK_MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#dcdcd6"

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK_MUTED,
    "text.color": INK, "xtick.color": INK_MUTED, "ytick.color": INK_MUTED,
    "grid.color": GRID, "grid.linewidth": 0.6, "axes.grid": True,
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.dpi": 120, "font.size": 9, "lines.linewidth": 2.0,
})

# Columns every game logs; anything else in a log is game-specific.
BASE = {"episode", "episode_reward", "moving_avg", "avg_loss", "max_q", "epsilon",
        "frames", "training_steps", "replay_buffer_size", "timestamp"}


def list_runs() -> pd.DataFrame:
    """Every run on disk, newest first, with a few headline numbers."""
    rows = []
    for log in sorted(RUNS.glob("*/*/training_log.csv")):
        d = log.parent
        try:
            df = pd.read_csv(log)
        except Exception:
            continue
        if df.empty:
            continue
        cfg = {}
        cfen = d / "config.json"
        if cfen.exists():
            cfg = json.loads(cfen.read_text())
        rows.append({
            "run": str(d.relative_to(RUNS)),
            "game": cfg.get("game", d.parent.name),
            "model": cfg.get("model", "?"),
            "episodes": len(df),
            "best_reward": df["episode_reward"].max(),
            "final_100_reward": df["episode_reward"].tail(100).mean().round(2),
            "max_q_peak": df["max_q"].max() if "max_q" in df else float("nan"),
        })
    return pd.DataFrame(rows).sort_values("run", ascending=False).reset_index(drop=True)


def load(run: str) -> tuple[pd.DataFrame, dict]:
    """Load one run's log and the config that produced it."""
    d = RUNS / run
    df = pd.read_csv(d / "training_log.csv")
    cfg = json.loads((d / "config.json").read_text()) if (d / "config.json").exists() else {}
    return df, cfg


def game_columns(df: pd.DataFrame) -> list[str]:
    """The adapter's own columns — numeric ones only."""
    return [c for c in df.columns
            if c not in BASE and pd.api.types.is_numeric_dtype(df[c])]


def smooth(s: pd.Series, window: int = 20) -> pd.Series:
    return s.rolling(window, min_periods=1).mean()


def _finish(ax, title, xlabel="episode", ylabel=None, legend=True):
    ax.set_title(title, color=INK, fontsize=10, loc="left", pad=8)
    ax.set_xlabel(xlabel)
    if ylabel:
        ax.set_ylabel(ylabel)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=6))
    if legend and ax.get_legend_handles_labels()[0]:
        ax.legend(frameon=False, fontsize=8, labelcolor=INK_MUTED)


print(f"runs directory: {RUNS}")

## What runs exist

In [ ]:
runs = list_runs()
runs

## Pick a run

Set `RUN` to any value from the `run` column above.

In [ ]:
RUN = runs.iloc[0]["run"] if len(runs) else None       # newest by default
df, cfg = load(RUN)

a = cfg.get("args", {})
print(f"{RUN}")
print(f"  {cfg.get('game','?')} / {cfg.get('model','?')} / state {cfg.get('state','?')}"
      f"   commit {cfg.get('git_commit','?')}")
print(f"  episodes {len(df)} of {a.get('num_episodes','?')}"
      f" | lr {a.get('learning_rate')} | eps_decay {a.get('epsilon_decay')}"
      f" | reward_clip {a.get('reward_clip')} | max_frames {a.get('max_frames')}")
print(f"  game-specific columns: {game_columns(df)}")
df.tail(3)

## Learning curve

Raw episode reward is noisy by nature, so the smoothed line carries the signal
and the raw series sits behind it for context.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.plot(df["episode"], df["episode_reward"], color=SERIES[0], alpha=0.22, linewidth=1.0)
ax.plot(df["episode"], smooth(df["episode_reward"]), color=SERIES[0],
        label="episode reward (20-ep mean)")
ax.axhline(0, color=GRID, linewidth=1.0, zorder=0)
_finish(ax, f"Reward — {RUN}", ylabel="reward")
plt.tight_layout(); plt.show()

## What the agent actually did

One panel per adapter column, because they are on different scales — a shared
axis would make the smaller ones unreadable, and a second y-axis is never the
answer.

In [ ]:
cols = game_columns(df)
if cols:
    fig, axes = plt.subplots(len(cols), 1, figsize=(9, 2.5 * len(cols)), sharex=True)
    axes = [axes] if len(cols) == 1 else list(axes)
    for ax, col, colour in zip(axes, cols, SERIES[1:]):
        ax.plot(df["episode"], df[col], color=colour, alpha=0.22, linewidth=1.0)
        ax.plot(df["episode"], smooth(df[col]), color=colour, label=f"{col} (20-ep mean)")
        _finish(ax, col, xlabel="" if ax is not axes[-1] else "episode", ylabel=col)
    plt.tight_layout(); plt.show()
else:
    print("this log has no game-specific columns")

## Stability

`max_q` is the divergence tripwire. With reward clipping on, a legitimate
\|Q\| cannot exceed `reward_clip / (1 - discount_factor)` — the dashed line.
Orders of magnitude above it means the value function is running away, which
is exactly how both Mario divergences were eventually spotted.

Log scale, because a healthy run and a diverging one differ by 10^15.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))

if "max_q" in df:
    axes[0].plot(df["episode"], df["max_q"].clip(lower=1e-6), color=SERIES[1], label="max |Q|")
    clip = (cfg.get("args") or {}).get("reward_clip") or 1.0
    gamma = (cfg.get("args") or {}).get("discount_factor") or 0.99
    if clip > 0:
        ceiling = clip / (1 - gamma)
        axes[0].axhline(ceiling, color=INK_MUTED, linestyle="--", linewidth=1.2,
                        label=f"ceiling {ceiling:.0f}")
    axes[0].set_yscale("log")
_finish(axes[0], "Value magnitude", ylabel="max |Q| (log)")

axes[1].plot(df["episode"], df["avg_loss"].clip(lower=1e-9), color=SERIES[2], label="avg loss")
axes[1].set_yscale("log")
_finish(axes[1], "Loss", ylabel="loss (log)")

plt.tight_layout(); plt.show()

## Exploration and episode length

Epsilon should actually reach `epsilon_min` by the end of the run — if the
curve flattens well above it, the decay is mistuned for `num_episodes` and the
agent spends the whole run partly random.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))

axes[0].plot(df["episode"], df["epsilon"], color=SERIES[3], label="epsilon")
floor = (cfg.get("args") or {}).get("epsilon_min")
if floor:
    axes[0].axhline(floor, color=INK_MUTED, linestyle="--", linewidth=1.2,
                    label=f"epsilon_min {floor}")
_finish(axes[0], "Exploration", ylabel="epsilon")

cap = (cfg.get("args") or {}).get("max_frames")
axes[1].plot(df["episode"], df["frames"], color=SERIES[4], alpha=0.25, linewidth=1.0)
axes[1].plot(df["episode"], smooth(df["frames"]), color=SERIES[4], label="frames (20-ep mean)")
if cap:
    axes[1].axhline(cap, color=INK_MUTED, linestyle="--", linewidth=1.2,
                    label=f"max_frames {cap}")
    hit = int((df["frames"] >= cap).sum())
    print(f"episodes truncated by max_frames: {hit} of {len(df)} ({hit/len(df):.0%})")
_finish(axes[1], "Episode length", ylabel="frames")

plt.tight_layout(); plt.show()

## Compare runs

Colour follows the run, assigned in fixed order — adding or removing a run
never repaints the others.

In [ ]:
# Any subset of runs["run"]. Defaults to every run of the same game.
COMPARE = [r for r in runs["run"] if runs.set_index("run").loc[r, "game"] == cfg.get("game")][:6]

metric = "episode_reward"        # or any column the runs share, e.g. "score", "kills", "max_x"

fig, ax = plt.subplots(figsize=(9, 3.6))
for run, colour in zip(COMPARE, SERIES):
    d, _ = load(run)
    if metric not in d:
        continue
    label = run.split("/")[-1][:28]
    ax.plot(d["episode"], smooth(d[metric]), color=colour, label=label)
    # direct-label the line end so identity is never colour-alone
    if len(d):
        ax.annotate(label, (d["episode"].iloc[-1], smooth(d[metric]).iloc[-1]),
                    xytext=(4, 0), textcoords="offset points",
                    color=INK_MUTED, fontsize=7, va="center")
_finish(ax, f"{metric} across runs — {cfg.get('game','?')}", ylabel=metric)
plt.tight_layout(); plt.show()

## Table view

The numbers behind the charts, so nothing depends on reading a colour.

In [ ]:
summary = df[["episode", "episode_reward", "moving_avg", "epsilon", "frames"] + game_columns(df)]
summary.tail(20)